# 📊 01 – EDA: GTZAN Adatfeltárás

**Cél:** Az adathalmaz mélyebb megismerése mielőtt bármilyen modellt építenénk.
Megvizsgáljuk:
- az osztályeloszlást (kiegyensúlyozott-e az adat?)
- az adatintegritást (van-e sérült, rövid vagy furcsa fájl?)
- a spektrogramokat (hogyan néznek ki a műfajok vizuálisan?)
- a CSV feature-öket (mit tartalmaz a Baseline ML-hez előkészített tábla?)

> 💡 **Minden megfigyelést jegyezz fel a vault `01_EDA.md` fájlban!**

---
## 🧠 Elméleti háttér – Mi ez a notebook?

Ez az **EDA (Exploratory Data Analysis)** notebook: az adatfeltárás lépései.  
Az EDA célja, hogy **mielőtt modellt építenénk**, megértsük:

- Milyen az adat szerkezete?
- Vannak-e hibák, hiányok?
- Hogyan néz ki az adat vizuálisan?
- Milyen jellemzők (feature-ök) állnak rendelkezésre?

A GTZAN adathalmazban **10 zenei műfaj** 100-100 hangfájlja található.  
Az EDA segít meggyőződni arról, hogy az adat valóban felhasználható modellezésre.


In [5]:
# ── Importok ──────────────────────────────────────────────────────────────────
import os
import librosa                   # Audio feldolgozó könyvtár: betöltés, spektrogram, feature extraction
import librosa.display           # Librosa vizualizációs modul (spektrogram rajzoláshoz)
import numpy as np               # Numerikus tömb műveletek (mátrixok, számítások)
import pandas as pd              # Táblázatos adatkezelés (DataFrame, CSV olvasás)
import matplotlib.pyplot as plt  # Általános célú vizualizációs könyvtár
import seaborn as sns            # Statisztikai vizualizáció (szebb alapstílus, boxplot, heatmap)
from pathlib import Path         # Modern, objektumorientált fájlútvonal-kezelés

# ── Útvonalak: notebook helyétől független, automatikus felderítés ─────────────
# Path.cwd() megadja, melyik mappából fut a notebook
# Ha 'notebooks' mappából fut → a projekt gyökere egy szinttel feljebb van
# Ha a projekt gyökeréből fut → marad ott
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

DATA_DIR   = PROJECT_DIR / 'data' / 'genres_original'  # .wav fájlok helye
OUTPUT_DIR = PROJECT_DIR / 'data'                       # kimeneti képek helye
CSV_30     = PROJECT_DIR / 'data' / 'features_30_sec.csv'  # 30 mp-es szegmensek feature táblája
CSV_3      = PROJECT_DIR / 'data' / 'features_3_sec.csv'   # 3 mp-es szegmensek feature táblája

# Mintavételi frekvencia: 22050 Hz = audio iparági szabvány
# Ez azt jelenti, hogy másodpercenként 22050 mintát tárolunk a hangból
SR = 22050

# ── Ellenőrzés ────────────────────────────────────────────────────────────────
print('Projekt gyökér :', PROJECT_DIR)
print('DATA_DIR létezik:', DATA_DIR.exists())
print('CSV_30 létezik  :', CSV_30.exists())

# GENRES: betöltjük az összes műfaj-mappát, ábécé sorrendben
# d.is_dir() szűri ki a mappákat (kizárva pl. .DS_Store rejtett fájlokat)
GENRES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print('Talált műfajok :', GENRES)
print('Összes fájl    :', sum(len(list((DATA_DIR/g).glob('*.wav'))) for g in GENRES))

Projekt gyökér : /home/feri/Asztal/gtzan-music-genre-recognition
DATA_DIR létezik: True
CSV_30 létezik  : True
Talált műfajok : ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Összes fájl    : 1000


## 1. Osztályeloszlás

Először megnézzük, hogy **kiegyensúlyozott-e az adathalmaz** – azaz minden műfajból ugyanannyi hangfájl áll-e rendelkezésre.
Ha nem egyensúlyos, a modell hajlamos lesz a többségi osztályokat jobban megtanulni.

---
### 🧠 Elméleti háttér – Osztályeloszlás

**Miért fontos ez?**

A gépi tanulási modellek érzékenyek az **osztályegyensúlyra**.  
Ha az egyik műfajból 150 fájl van, egy másikból csak 50, a modell megtanul a többségi osztályra torzítani –
azaz mindig azt "tippeli", amire több példát látott.

A GTZAN **szándékosan kiegyensúlyozott**: minden műfajból pontosan 100 fájl (ideális eset).  
Az `axhline(100)` piros vonala ezt az elvárást jelöli – ha valamelyik műfaj alatta van, az probléma.

| Fogalom | Jelentés |
|---------|----------|
| **Kiegyensúlyozott adat** | Minden osztályból ugyanannyi minta |
| **Kiegyensúlyozatlan adat** | Egyes osztályokból jóval több/kevesebb minta → torzítja a modellt |
| **`value_counts()`** | Megszámolja, hány sor tartozik minden osztályhoz |


In [ ]:
# Dictionary comprehension: minden műfajhoz megszámoljuk a .wav fájlokat
# (DATA_DIR/g) összerakja az útvonalat pl: data/genres_original/blues
# .glob('*.wav') listázza az összes .wav kiterjesztésű fájlt
counts = {g: len(list((DATA_DIR/g).glob('*.wav'))) for g in GENRES}

# Átalakítjuk DataFrame-mé, hogy szebb legyen megjeleníteni és rajzolni
df_counts = pd.DataFrame(counts.items(), columns=['Műfaj', 'Fájlok'])

# Ábrázolás: oszlopdiagram a fájlszámokról
fig, ax = plt.subplots(figsize=(10, 4))  # (szélesség, magasság) inch-ben
sns.barplot(data=df_counts, x='Műfaj', y='Fájlok', ax=ax, palette='viridis')

# Vízszintes vonal az elvárt 100-as szintnél – ezzel látjuk a hiányokat
ax.axhline(100, color='red', linestyle='--', label='Elvárt: 100 fájl/műfaj')
ax.set_title('Osztályeloszlás – GTZAN adathalmaz')
ax.legend()
plt.tight_layout()  # automatikusan rendezi el az elemeket, hogy ne fedjék egymást
plt.show()

# Szöveges összefoglaló táblázat
print(df_counts.to_string(index=False))

## 2. Adatintegritás – hossz és mintavételi frekvencia ellenőrzés

Minden fájlnál ellenőrizzük:
- **Hossz**: a GTZAN fájlok elvileg mind pontosan 30 másodpercesek
- **Mintavételi frekvencia (SR)**: elvileg mind 22050 Hz
- **Olvashatóság**: nem sérült-e a fájl?

> ⚠️ Ismert hiba: `jazz.00054.wav` – ez egy dokumentált sérült fájl a GTZAN-ban, ha megjelenik, ne töröld!

---
### 🧠 Elméleti háttér – Adatintegritás

#### Mintavételi frekvencia (Sample Rate, SR = 22050 Hz)

A digitális hangot úgy tároljuk, hogy másodpercenként rögzítjük a hanghullám amplitúdóját.  
Az SR azt mondja meg, hányszor másodpercenként:

- **22050 Hz** = másodpercenként 22 050 mintapont
- Ez az audio-feldolgozásban legelterjedtebb szabvány (CD minőség: 44100 Hz)
- Az SR konzisztenciája azért fontos, mert ha különböző fájlokat különböző SR-rel töltünk be,
  a feature-öket nem lehet összehasonlítani

#### Fájlhossz (Duration = 30 mp)

A GTZAN minden fájlja elvileg pontosan 30 másodperces.  
Ha valamelyik rövidebb (pl. 28 mp), az jelezhet:
- sérült fájlt
- csonkított letöltést
- kódolási problémát

> ⚠️ **Ismert kivétel:** `jazz.00054.wav` – ez egy dokumentált sérült fájl a GTZAN adathalmazban.  
> A kutatói közösség is tudja, általában megtartják (csak kizárják az eredmények értékelésekor).

#### `librosa.get_duration()` vs `librosa.load()`

| Függvény | Mit csinál | Sebesség |
|----------|-----------|----------|
| `get_duration(path=...)` | Csak a fejlécet olvassa (metaadat) | ⚡ Gyors |
| `load(sr=None, duration=1.0)` | Az első 1 mp hangadatot olvassa be (SR lekéréséhez) | 🐢 Kicsit lassabb |

Ezért a hossz ellenőrzésnél `get_duration()`-t, az SR ellenőrzésnél `load(duration=1.0)`-t használunk.


---
### 🧠 Elméleti háttér – `describe()` statisztikák értelmezése

A pandas `.describe()` függvény egy sor **leíró statisztikát** számol egyszerre.

| Statisztika | Leírás | Mit mond el? |
|---|---|---|
| `count` | Hiányzó értékek nélküli sorok száma | Teljes-e az adat? |
| `mean` | Számtani átlag | Az általános szint |
| `std` | Szórás – az értékek átlagtól való eltérése | Mennyire homogén az adat? |
| `min` | Legkisebb érték | Outlier, hibás adat detektálása |
| `25%` | 1. kvartilis (Q1) | Az adatok 25%-a ez alatt van |
| `50%` | Medián (Q2) | Középső érték – nem érzékeny kiugró értékekre |
| `75%` | 3. kvartilis (Q3) | Az adatok 75%-a ez alatt van |
| `max` | Legnagyobb érték | Maximális terjedelem, kiugró értékek |

#### IQR – Interkvartilis terjedelem

`IQR = Q3 − Q1` – a középső 50% szélességét méri.  
Outlier-ek szűrése: az `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` tartományon kívüli értékek gyanúsak.

#### Mean vs Medián – mikor melyik jobb?

| Helyzet | Javasolt mutató |
|---|---|
| Szimmetrikus eloszlás | `mean` |
| Kiugró értékek (outlier) jelenléte | `median` (50%) |
| `mean` >> `median` | Jobbra ferde eloszlás (néhány nagy érték húzza fel az átlagot) |

#### A GTZAN kontextusban

Az alábbi cellában a **hangfájlok hosszát** vizsgáljuk műfajonként.  
Ideális esetben minden fájl pontosan **30.0 másodperc** (`count=100`, `std≈0`).  
Ha a `min` lényegesen kisebb 30-nál, sérült vagy csonkított fájl van az adathalmazban.


In [ ]:
records = []  # Ide gyűjtjük az összes fájl metaadatait
errors  = []  # Ide gyűjtjük a beolvasási hibákat

for genre in GENRES:
    # Végigmegyünk minden .wav fájlon az adott műfaj mappájában
    for wav_path in sorted((DATA_DIR/genre).glob('*.wav')):
        try:
            # librosa.get_duration() csak a fejlécet olvassa be, nem a teljes fájlt
            # Ez sokkal gyorsabb, mint a teljes audio betöltése
            duration = librosa.get_duration(path=str(wav_path))

            # Betöltjük az audio első 1 másodpercét a mintavételi frekvencia lekéréséhez
            # sr=None → megtartja az eredeti SR-t (nem resampel)
            # duration=1.0 → csak az első másodpercet olvassa be (gyorsabb)
            y, sr = librosa.load(str(wav_path), sr=None, duration=1.0)

            records.append({
                'genre': genre,
                'file': wav_path.name,
                'duration': round(duration, 2),
                'sr': sr
            })
        except Exception as e:
            # Ha a fájl sérült vagy olvashatatlan, ide kerül
            errors.append({'file': str(wav_path), 'error': str(e)})

# DataFrame-mé alakítjuk az összegyűjtött adatokat
df_meta = pd.DataFrame(records)

print('=== Hibás / olvashatatlan fájlok ===')
print(pd.DataFrame(errors) if errors else 'Nincs hibás fájl ✅')
print()

# groupby('genre') → csoportosítás műfaj szerint
# ['duration'].describe() → min, max, mean, std statisztikák
print('=== Hossz statisztika (másodperc) ===')
print(df_meta.groupby('genre')['duration'].describe().round(2))
print()

# Mintavételi frekvenciák eloszlása – ha nem mind 22050, az gyanús
print('=== Mintavételi frekvenciák ===')
print(df_meta['sr'].value_counts())

In [ ]:
# Gyanús fájlok kiszűrése

EXPECTED_DUR = 30.0    # elvárt hossz: 30 másodperc
EXPECTED_SR  = 22050   # elvárt mintavételi frekvencia

# Boolean indexelés: csak azokat a sorokat tartjuk meg,
# ahol a hossz 1 másodpercnél rövidebb (< 29s) VAGY az SR eltér
suspicious = df_meta[
    (df_meta['duration'] < EXPECTED_DUR - 1) |
    (df_meta['sr'] != EXPECTED_SR)
]

print(f'Gyanús fájlok száma: {len(suspicious)}')
if len(suspicious) > 0:
    print(suspicious[['genre','file','duration','sr']])
else:
    print('Minden fájl megfelelő hosszú és SR-ű ✅')

## 3. Audio metaadatok részletes elemzése

Az előző szekcióban csak a **hosszt** és a **mintavételi frekvenciát** ellenőriztük.  
Most kibővítjük a `df_meta` táblát az alábbi plusz mezőkkel:

| Mező | Magyar neve | Mit mér? |
|------|-------------|----------|
| `file_size_kb` | Fájlméret (KB) | A WAV fájl fizikai mérete lemezen |
| `channels` | Csatornák száma | 1 = mono, 2 = stereo |
| `bit_depth` | Bitmélység | Mintánkénti bitek száma (pl. 16-bit PCM) |
| `n_samples` | Minták száma | `duration × sr` – a valódi mintaszám |
| `format` | Fájlformátum | WAV altípus (pl. PCM_16) |


---
### 🧠 Elméleti háttér – Audio fájl metaadatok

#### Fájlméret és bitmélység

Egy WAV fájl elméleti mérete:

> `fájlméret ≈ sr × duration × channels × (bit_depth / 8)` bájt

Például **22050 Hz, 30 s, mono, 16-bit** esetén:  
`22050 × 30 × 1 × 2 = 1 323 000 bájt ≈ 1292 KB`

#### Csatornák (channels)

- **Mono (1):** egyetlen hangjel – a GTZAN alapértelmezetten mono
- **Stereo (2):** bal + jobb – ha valamelyik fájl stereo, az eltérés is vizsgálandó!

#### Bitmélység (bit depth)

| Bitmélység | Dinamikatartomány | Jellemző felhasználás |
|---|---|---|
| 8-bit (PCM_U8) | ~48 dB | régi hangformátumok |
| **16-bit (PCM_16)** | **~96 dB** | **CD minőség – GTZAN standard** |
| 24-bit (PCM_24) | ~144 dB | stúdió felvételek |

#### `soundfile` könyvtár

`soundfile.info(path)` csak a fejlécet olvassa be (gyors, nem tölti be az egész audiót).  
Visszaadja: `samplerate`, `channels`, `frames`, `subtype` (pl. `PCM_16`), `format` (`WAV`)


In [ ]:
import soundfile as sf

extra_records = []

for genre in GENRES:
    for wav_path in sorted((DATA_DIR / genre).glob('*.wav')):
        try:
            info         = sf.info(str(wav_path))
            file_size_kb = wav_path.stat().st_size / 1024
            extra_records.append({
                'file':         wav_path.name,
                'genre':        genre,
                'file_size_kb': round(file_size_kb, 1),
                'channels':     info.channels,
                'bit_depth':    info.subtype,
                'n_samples':    info.frames,
                'format':       info.format,
            })
        except Exception as e:
            print(f'  ⚠ Hiba: {wav_path.name} → {e}')

df_audio = pd.DataFrame(extra_records)
df_meta_full = df_meta.merge(df_audio, on=['file', 'genre'], how='left')

print(f'Kibővített tábla: {df_meta_full.shape}')
display(df_meta_full.head(3))

print('\n── Csatornák ──')
print(df_audio['channels'].value_counts())
print('\n── Bitmélység ──')
print(df_audio['bit_depth'].value_counts())
print('\n── Fájlméret statisztika (KB) ──')
print(df_audio['file_size_kb'].describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Audio fájl metaadatok – GTZAN', fontsize=13)

genre_order = sorted(GENRES)
sns.boxplot(data=df_meta_full, x='genre', y='file_size_kb', order=genre_order, ax=axes[0])
axes[0].set_xlabel('Műfaj')
axes[0].set_ylabel('KB')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Fájlméret műfajonként (KB)')

ch_counts = df_audio['channels'].value_counts().sort_index()
ch_labels = {1: 'Mono', 2: 'Stereo'}
axes[1].bar([ch_labels.get(c, str(c)) for c in ch_counts.index],
            ch_counts.values, color=['#2196F3', '#FF5722'])
axes[1].set_title('Csatornák eloszlása')
axes[1].set_ylabel('Fájlok száma')
for bar in axes[1].patches:
    axes[1].annotate(f'{int(bar.get_height())}',
        (bar.get_x() + bar.get_width()/2, bar.get_height()),
        ha='center', va='bottom', fontsize=11)

bd_counts = df_audio['bit_depth'].value_counts()
axes[2].bar(bd_counts.index, bd_counts.values, color='#4CAF50')
axes[2].set_title('Bitmélység eloszlása')
axes[2].set_ylabel('Fájlok száma')
axes[2].tick_params(axis='x', rotation=30)
for bar in axes[2].patches:
    axes[2].annotate(f'{int(bar.get_height())}',
        (bar.get_x() + bar.get_width()/2, bar.get_height()),
        ha='center', va='bottom', fontsize=11)

plt.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_DIR / 'eda_audio_metadata.png', dpi=100, bbox_inches='tight')
plt.show()

unique_ch = df_audio['channels'].unique()
print(f'Csatornák: {sorted(unique_ch)} → {"✅ Mind mono" if list(unique_ch)==[1] else "⚠ Vegyes!"}')
print(f'Bitmélység: {list(df_audio["bit_depth"].unique())}')
print(f'Átl. fájlméret: {df_audio["file_size_kb"].mean():.1f} KB')
print(f'Min: {df_audio["file_size_kb"].min():.1f} KB  ({df_audio.loc[df_audio["file_size_kb"].idxmin(), "file"]})')
print(f'Max: {df_audio["file_size_kb"].max():.1f} KB  ({df_audio.loc[df_audio["file_size_kb"].idxmax(), "file"]})')


## 4. Waveform és Mel-spektrogram vizsgálat

**Waveform**: az audiojel amplitúdója az idő függvényében – megmutatja a dinamikát.

**Mel-spektrogram**: a frekvenciatartalom időbeli változása, **Mel-skálán** ábrázolva.
- X tengely: idő
- Y tengely: frekvencia (Mel-skála – az emberi halláshoz igazított logaritmikus skála)
- Szín: intenzitás decibelben (dB)

Ez lesz a CNN bemenete – lényegében hangfájlból kép lesz!

---
### 🧠 Elméleti háttér – Waveform és Mel-spektrogram

#### 1. Waveform (Hanghullám)

A waveform az audiojel **nyers ábrázolása**: az X tengelyen az idő, az Y tengelyen az amplitúdó (hangerő).

- **Nagy amplitúdó** = hangos rész (pl. dob ütés)
- **Kis amplitúdó** = csendes rész (pl. szünet)
- A waveform nem mutatja, **milyen frekvenciák** vannak jelen – erre kell a spektrogram

---

#### 2. Fourier-transzformáció (röviden)

Egy hangot fel lehet bontani egyszerű szinuszok összegére – ez a Fourier-transzformáció alapgondolata.  
Eredménye: megtudjuk, hogy **melyik frekvencia mekkora erővel van jelen** egy adott pillanatban.

---

#### 3. Mel-skála

Az emberi fül **nem lineárisan** érzékeli a hangmagasságokat.  
Alacsony frekvenciáknál jól halljuk a különbséget (200 Hz vs 300 Hz),  
de magasaknál ez nehezebb (8000 Hz vs 9000 Hz – alig hallható különbség).

A **Mel-skála** logaritmikus tömörítést alkalmaz, hogy az emberi halláshoz igazítsa a frekvenciatengelyt.

| Lineáris Hz | Mel értéke |
|-------------|-----------|
| 100 Hz | ~150 Mel |
| 1000 Hz | ~1000 Mel |
| 4000 Hz | ~2146 Mel |
| 8000 Hz | ~2840 Mel |

---

#### 4. Mel-spektrogram

A Mel-spektrogram egy **2D kép**:
- **X tengely**: idő
- **Y tengely**: frekvencia Mel-skálán
- **Szín**: intenzitás decibelben (dB) – sötét = gyenge, élénk/piros = erős

**Miért dB?**  
Az emberi fül logaritmikusan érzékeli a hangerőt. A `power_to_db()` logaritmikus skálára konvertál,
hogy a spektrogram vizuálisan informatívabb legyen.

```python
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
# n_mels=128 → 128 frekvencia-sáv (a kép magassága lesz)
# fmax=8000  → csak 8000 Hz-ig nézzük (zenei tartomány)

mel_db = librosa.power_to_db(mel, ref=np.max)
# ref=np.max → a maximumhoz képest normalizál (relatív dB értékek)
# Tipikus tartomány: 0 dB (legerősebb rész) ... -80 dB (szinte csend)
```

**Miért ez lesz a CNN bemenete?**  
A CNN képfelismerésre van optimalizálva. Ha a hangot képpé alakítjuk (Mel-spektrogram),
a CNN ugyanúgy tanulhat zenei mintázatokat, ahogy képeken tanul vízszintes/függőleges éleket.


In [ ]:
fig, axes = plt.subplots(len(GENRES), 2, figsize=(15, len(GENRES)*2.5),
                         gridspec_kw={'width_ratios': [1, 1]})
fig.suptitle('Waveform + Mel-spektrogram – 1 minta műfajonként', fontsize=14)

for i, genre in enumerate(GENRES):
    sample = sorted((DATA_DIR/genre).glob('*.wav'))[0]
    y, sr = librosa.load(str(sample), sr=SR, duration=10.0)

    # Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[i, 0])
    axes[i, 0].set_title(f'{genre} – waveform', fontsize=9)
    axes[i, 0].set_xlabel('')

    # Mel-spektrogram
    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img    = librosa.display.specshow(
        mel_db, sr=sr,
        x_axis='time', y_axis='mel',
        ax=axes[i, 1], fmax=8000
    )
    axes[i, 1].set_title(f'{genre} – Mel-spektrogram', fontsize=9)

# Colorbar: a fig jobb szélén, egy önálló helyen
# fig.add_axes([bal, lent, szélesség, magasság]) – értékek 0–1 között, az egész fig-hez képest
cbar_ax = fig.add_axes([0.92, 0.05, 0.015, 0.88])  # vékony sáv a jobb szélen
fig.colorbar(img, cax=cbar_ax, format='%+2.0f dB')

# tight_layout-ot csak a bal részig alkalmazzuk, helyet hagyva a colorbar-nak
plt.tight_layout(rect=[0, 0, 0.91, 1])

out_path = OUTPUT_DIR / 'eda_mel_spectrograms.png'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Kép mentve: {out_path}')

## 5. CSV feature-ök vizsgálata (Baseline ML-hez)

A GTZAN dataset tartalmaz egy előre kiszámított feature táblázatot (`features_30_sec.csv`).
Ez olyan jellemzőket tartalmaz, mint:
- **MFCC** (Mel-Frequency Cepstral Coefficients): a hang "timbréjét" leíró együtthatók
- **Chroma**: a 12 zenei hangmagasság erősségei
- **Tempo**: becsült BPM
- **Spectral features**: spektrális centroid, bandwidth, stb.

Ezt a táblázatot fogjuk használni a **Szint 1-2 Baseline ML modellekhez** (Random Forest, SVM stb.)

---
### 🧠 Elméleti háttér – CSV feature-ök (előkészített jellemzők)

A `features_30_sec.csv` fájlban az összes hangfájlból **előre kiszámított** numerikus jellemzők vannak.
Ez a "klasszikus" ML megközelítés: nem a nyers hangot, hanem a belőle kinyert számokat adjuk a modellnek.

---

#### MFCC – Mel-Frequency Cepstral Coefficients

Az MFCC-k a **hang "timbréjét"** (hangszínét) írják le – olyan jellemzőket, amelyek alapján
megkülönböztethető pl. egy gitár és egy zongora ugyanazon hangjegyénél.

Kiszámítási lépések:
1. Mel-spektrogram kiszámítása
2. Logaritmizálás (dB)
3. DCT (diszkrét koszinusz transzformáció) → kompakt reprezentáció

Tipikusan **20 MFCC együtthatót** számolunk, mindegyikből `_mean` és `_var` értéket tárolunk:
- `mfcc1_mean` = az 1. MFCC átlaga az egész hangfájlon
- `mfcc1_var`  = az 1. MFCC varianciája (mennyire változik az idő során)

---

#### Chroma – Hangmagasság-profilok

A **Chroma feature** a 12 zenei hangmagasság (C, C#, D, D#, E, F, F#, G, G#, A, A#, B)
relatív erősségét méri. Hasznos az akkordok és hangnem felismeréséhez.

---

#### Spectral Centroid (Spektrális Centroid)

A spektrum "súlypontja" – megmutatja, hogy a hang energiája inkább az alacsony vagy magas frekvenciáknál van.

- **Alacsony érték** (~1500 Hz) → mélyebb hang (pl. blues, jazz)
- **Magas érték** (~3000+ Hz) → fényesebb hang (pl. metal, pop)

---

#### Összefoglalás – mit mihez használunk

| Feature | Méri | Hasznos erre |
|---------|------|-------------|
| MFCC | Hangszín / timbré | Hangszer- és műfajfelismerés |
| Chroma | Hangmagasság-profil | Akkord, hangnem |
| Tempo | Ritmus / BPM | Tánczenék megkülönböztetése |
| Spectral Centroid | "Fényesség" | Mély vs. magas hangzás |
| RMS Energy | Hangerő / energia | Dinamika |
| Spectral Bandwidth | Hangzás teltségi foka | Rock vs. klasszikus |


In [ ]:
# CSV fájl betöltése pandas DataFrame-be
# CSV_30 az első cellában definiált abszolút útvonal
df_feat = pd.read_csv(CSV_30)

# Alapstatisztikák
print('DataFrame alakja (sorok, oszlopok):', df_feat.shape)
print()
print('Első 10 oszlop neve:')
print(df_feat.columns.tolist()[:10], '...')
print()

# Osztályeloszlás: value_counts() megszámolja az egyedi értékeket
print('Osztályeloszlás (label oszlop):')
print(df_feat['label'].value_counts())
print()

# Hiányzó értékek keresése
# isnull() True-t ad hiányzó értéknél, .sum().sum() az összes True-t összegzi
missing = df_feat.isnull().sum().sum()
print(f'Hiányzó értékek száma: {missing} db', '✅' if missing == 0 else '⚠️')
print()

# Első 3 sor megjelenítése
df_feat.head(3)

---
### 🧠 Elméleti háttér – Korrelációs mátrix

#### Mi a korreláció?

A **Pearson-korreláció** megmutatja, hogy két változó mennyire "mozog együtt":

| Érték | Jelentés |
|-------|----------|
| **+1.0** | Teljesen pozitív korreláció (ha az egyik nő, a másik is nő) |
| **0.0** | Nincs összefüggés |
| **-1.0** | Teljesen negatív korreláció (ha az egyik nő, a másik csökken) |

#### Miért fontos a gépi tanulásban?

Ha két feature **erősen korrelál**, lényegében **ugyanazt mérik** – ez redundancia.

- **Sötét piros blokkok** → erős pozitív korreláció (redundáns feature-ök)
- **Sötét kék blokkok** → erős negatív korreláció
- **Fehér / halvány** → gyenge/nincs korreláció

**Mit lehet tenni erős korreláció esetén?**

- Eltávolítani az egyik redundáns feature-t (**feature selection**)
- Vagy **PCA** (Principal Component Analysis): a korrelált jellemzőket kevesebb, független komponensre vetítjük
- A Random Forest modelleknél ez kevésbé probléma – automatikusan kezeli


In [ ]:
# ── Feature korrelációs heatmap ────────────────────────────────────────────
# Megmutatja, hogy a feature-ök mennyire függnek össze egymással
# Erős korreláció (~1.0 vagy ~-1.0) = redundáns feature-ök

# select_dtypes(include=np.number) → csak numerikus oszlopokat vesz
# (kizárja pl. 'filename', 'label' szöveges oszlopokat)
numeric_cols = df_feat.select_dtypes(include=np.number).columns

# .corr() kiszámítja a Pearson-korrelációs mátrixot
corr = df_feat[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr,
    cmap='coolwarm',  # kék = negatív korreláció, piros = pozitív
    center=0,         # 0 korreláció = fehér szín
    ax=ax,
    xticklabels=False, yticklabels=False  # túl sok feature, feliratok olvashatatlanok lennének
)
ax.set_title('Feature korrelációs mátrix – összes numerikus jellemző')
plt.tight_layout()
plt.show()

---
### 🧠 Elméleti háttér – Boxplot (doboz-ábra)

#### Mit mutat a boxplot?

| Elem | Jelentés |
|------|----------|
| **Doboz alja (Q1)** | 25. percentilis – az értékek 25%-a ez alatt van |
| **Középső vonal** | Medián (50. percentilis) |
| **Doboz teteje (Q3)** | 75. percentilis – az értékek 75%-a ez alatt van |
| **Bajuszok (whiskers)** | Q1 − 1.5×IQR ... Q3 + 1.5×IQR tartomány |
| **Pontok a bajuszon túl** | Kiugró értékek (outlier-ek) |

**IQR (Interquartile Range)** = Q3 − Q1 → a középső 50% terjedelme

#### Hogyan értelmezzük a műfaj-boxplotot?

- **Ha a dobozok NEM fedik át egymást** → a feature jól elkülöníti a műfajokat → **hasznos feature** ✅
- **Ha a dobozok teljesen átfednek** → a feature nem segít megkülönböztetni a műfajokat → **gyenge feature** ❌

**Tempo** esetén pl. várható, hogy a `disco` vagy `hiphop` magasabb BPM-et mutat, mint a `classical`.  
A boxplot alapján eldönthetjük, hogy érdemes-e egy feature-t megtartani a modellhez.


In [ ]:
# ── Boxplot: jellemzők eloszlása műfajonként ───────────────────────────────
# Ha a dobozok kevéssé fedik át egymást → a feature hasznos osztályozáshoz
# Ha a dobozok teljesen átfednek → a feature kevésbé informatív

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['tempo', 'chroma_stft_mean']):
    if col in df_feat.columns:
        sns.boxplot(
            data=df_feat,
            x='label',   # X tengelyen a műfajok
            y=col,        # Y tengelyen a feature értékei
            ax=ax,
            palette='Set2'
        )
        # X feliratok 45 fokos elforgatása az olvashatóságért
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_title(f'`{col}` eloszlása műfajonként')

plt.tight_layout()
plt.show()

## 6. EDA összefoglaló

Az alábbi cella kinyomtatja a legfontosabb megfigyeléseket.
**Másold be az eredményt a vault `01_EDA.md` fájlba!**

In [ ]:
print('='*50)
print('         EDA ÖSSZEFOGLALÓ – GTZAN')
print('='*50)
print(f'  Összes audio fájl:        {len(df_meta)}')
print(f'  Műfajok száma:            {len(GENRES)}')
print(f'  Olvasási hibák:           {len(errors)}')
print(f'  Gyanús fájlok:            {len(suspicious)}')
print(f'  Átlagos hossz:            {df_meta["duration"].mean():.1f} mp')
print(f'  CSV feature sorok:        {len(df_feat)}')
print(f'  CSV feature oszlopok:     {len(df_feat.columns)}')
print(f'  Hiányzó CSV értékek:      {df_feat.isnull().sum().sum()}')
print('='*50)
print()
print('✅ Következő lépés: 02_Baseline_ML.ipynb')
print('   → Random Forest, SVM, KNN a CSV feature-ökkel')